# Problème du Voyageur de Commerce (TSP)
## Résolution par Algorithme Génétique

---

### Description du Problème

On considère un ensemble de **15 villes** notées V = {C1, C2, ..., C15}.

Un voyageur doit :
- Partir d'une ville de départ
- Visiter **chaque ville exactement une seule fois**
- Revenir à la ville de départ

Chaque déplacement entre deux villes Ci et Cj est caractérisé par :
- Une **distance** dij
- Un **coût** cij

---

### Objectifs (Bi-objectif)

Le problème consiste à déterminer un **circuit hamiltonien** minimisant simultanément :

1. **f1(x)** = La distance totale parcourue = Σ dij
2. **f2(x)** = Le coût total du voyage = Σ cij

La fonction objectif combinée est :

**F(x) = f1(x) + f2(x)**

---

### Contraintes

- Chaque ville est visitée exactement une fois
- Le voyageur revient à la ville de départ
- Les sous-tournées sont interdites

---

## Importation des Bibliothèques

Nous utilisons les bibliothèques standard de Python :
- `csv` : Pour lire les fichiers de données
- `random` : Pour les opérations aléatoires de l'algorithme génétique
- `os` : Pour vérifier l'existence des fichiers

In [ ]:
import csv
import random
import os

---

## Paramètres de l'Algorithme Génétique

Les paramètres sont stockés dans un **dictionnaire** :

| Paramètre | Valeur | Description |
|-----------|--------|-------------|
| `taille_population` | 300 | Nombre de solutions par génération |
| `nb_generations` | 500 | Nombre maximal d'itérations |
| `taux_croisement` | 0.70 | Probabilité d'effectuer un croisement |
| `taux_mutation` | 0.15 | Probabilité d'effectuer une mutation |
| `taille_tournoi` | 5 | Nombre de participants pour la sélection |

In [ ]:
# Paramètres de l'algorithme génétique
PARAMS = {
    'taille_population': 300,    # 300 solutions aléatoires
    'nb_generations': 500,       # 500 itérations
    'taux_croisement': 0.70,     # 70% de croisements
    'taux_mutation': 0.15,       # 15% de mutations
    'taille_tournoi': 5          # Sélection par tournoi de 5
}

---

## 1. Lecture des Fichiers CSV

Les matrices des distances et des coûts sont fournies dans deux fichiers CSV :
- `distances.csv` : Contient la matrice des distances dij
- `cities.csv` : Contient la matrice des coûts cij

### Format des fichiers CSV

```
Ville,C1,C2,C3,...,C15
C1,0,12,18,...,50
C2,12,0,14,...,44
...
```

La première ligne contient les noms des villes, et chaque ligne suivante contient les distances/coûts depuis une ville vers toutes les autres.

In [ ]:
def lire_matrice_csv(chemin_fichier):
    """
    Lit une matrice depuis un fichier CSV.
    
    Paramètres:
        chemin_fichier: Chemin vers le fichier CSV
    
    Retourne:
        matrice: Liste de listes contenant les valeurs numériques
        villes: Liste des noms de villes
    """
    matrice = []
    villes = []
    
    with open(chemin_fichier, 'r', encoding='utf-8') as fichier:
        lecteur = csv.reader(fichier)
        
        # Première ligne = en-tête avec noms des villes
        entete = next(lecteur)
        villes = entete[1:]  # Ignorer la première colonne ("Ville")
        
        # Lire les lignes de données
        for ligne in lecteur:
            if ligne:
                # Convertir en entiers, ignorer la première colonne (nom de ville)
                valeurs = [int(v) for v in ligne[1:]]
                matrice.append(valeurs)
    
    return matrice, villes


def charger_donnees(fichier_distances, fichier_couts):
    """
    Charge les données complètes du problème TSP.
    
    Paramètres:
        fichier_distances: Chemin vers le fichier des distances
        fichier_couts: Chemin vers le fichier des coûts
    
    Retourne:
        Dictionnaire contenant les villes et les deux matrices
    """
    distances, villes = lire_matrice_csv(fichier_distances)
    couts, _ = lire_matrice_csv(fichier_couts)
    
    donnees = {
        'villes': villes,
        'distances': distances,
        'couts': couts
    }
    
    print(f"Données chargées avec succès: {len(villes)} villes")
    return donnees

In [ ]:
# Charger les données depuis les fichiers CSV
DONNEES = charger_donnees("distances.csv", "cities.csv")

print(f"Liste des villes: {DONNEES['villes']}")
print(f"\nExemple - Distance C1 vers C2: {DONNEES['distances'][0][1]}")
print(f"Exemple - Coût C1 vers C2: {DONNEES['couts'][0][1]}")

---

## Fonctions d'Accès aux Données

Ces fonctions permettent d'obtenir facilement la distance ou le coût entre deux villes en utilisant leurs noms.

In [ ]:
def obtenir_distance(ville_a, ville_b, donnees):
    """
    Retourne la distance entre deux villes.
    
    Paramètres:
        ville_a: Nom de la ville de départ (ex: 'C1')
        ville_b: Nom de la ville d'arrivée (ex: 'C5')
        donnees: Dictionnaire contenant les données du problème
    
    Retourne:
        La distance entre les deux villes
    """
    i = donnees['villes'].index(ville_a)
    j = donnees['villes'].index(ville_b)
    return donnees['distances'][i][j]


def obtenir_cout(ville_a, ville_b, donnees):
    """
    Retourne le coût entre deux villes.
    
    Paramètres:
        ville_a: Nom de la ville de départ
        ville_b: Nom de la ville d'arrivée
        donnees: Dictionnaire contenant les données du problème
    
    Retourne:
        Le coût entre les deux villes
    """
    i = donnees['villes'].index(ville_a)
    j = donnees['villes'].index(ville_b)
    return donnees['couts'][i][j]

---

## 2. Représentation d'une Solution

### Structure de données

Une solution est représentée par une **permutation des villes** (chemin fermé).

Nous utilisons un **dictionnaire** avec les clés suivantes :

| Clé | Type | Description |
|-----|------|-------------|
| `parcours` | Liste | Ordre de visite des villes [C1, C4, C7, ...] |
| `distance` | Entier | Distance totale f1(x) |
| `cout` | Entier | Coût total f2(x) |
| `score` | Entier | Valeur de F = f1 + f2 |

### Exemple de solution

```python
solution = {
    'parcours': ['C1', 'C4', 'C7', 'C10', 'C13', 'C15', 'C14', 'C12', 'C11', 'C9', 'C8', 'C6', 'C5', 'C3', 'C2'],
    'distance': 242,
    'cout': 301,
    'score': 543
}
```

Le parcours ci-dessus signifie :
- Départ de C1
- Visite successive de toutes les villes
- Retour final à C1

In [ ]:
def creer_solution_vide():
    """
    Crée une solution vide (sans parcours défini).
    
    Retourne:
        Dictionnaire représentant une solution vide
    """
    return {
        'parcours': [],
        'distance': 0,
        'cout': 0,
        'score': 0
    }


def creer_solution_aleatoire(donnees):
    """
    Génère une solution aléatoire.
    
    Le parcours est créé en mélangeant aléatoirement la liste des villes.
    
    Paramètres:
        donnees: Dictionnaire contenant les données du problème
    
    Retourne:
        Dictionnaire représentant une solution avec un parcours aléatoire
    """
    parcours = donnees['villes'].copy()  # Copier la liste des villes
    random.shuffle(parcours)              # Mélanger aléatoirement
    
    return {
        'parcours': parcours,
        'distance': 0,
        'cout': 0,
        'score': 0
    }


def copier_solution(solution):
    """
    Crée une copie indépendante d'une solution.
    
    Paramètres:
        solution: La solution à copier
    
    Retourne:
        Une nouvelle solution identique mais indépendante
    """
    return {
        'parcours': solution['parcours'].copy(),
        'distance': solution['distance'],
        'cout': solution['cout'],
        'score': solution['score']
    }

In [ ]:
# Exemple: Générer une solution aléatoire
exemple_solution = creer_solution_aleatoire(DONNEES)
print("Exemple de solution aléatoire:")
print(f"Parcours: {exemple_solution['parcours']}")

---

## 3. Évaluation d'une Solution

### Calcul des fonctions objectif

Pour une solution S = [C1 → C4 → C7 → ... → C2 → C1], on calcule :

**f1(S)** = d(1,4) + d(4,7) + ... + d(2,1)

**f2(S)** = c(1,4) + c(4,7) + ... + c(2,1)

**F(S)** = f1(S) + f2(S)

### Important

Le calcul inclut le **retour à la ville de départ** (le dernier trajet pour fermer le circuit).

In [ ]:
def evaluer_solution(solution, donnees):
    """
    Calcule les valeurs f1 (distance), f2 (coût) et F (score) d'une solution.
    
    Cette fonction modifie la solution en place en mettant à jour
    les champs 'distance', 'cout' et 'score'.
    
    Paramètres:
        solution: La solution à évaluer
        donnees: Dictionnaire contenant les données du problème
    """
    parcours = solution['parcours']
    distance_totale = 0
    cout_total = 0
    
    # Calculer pour chaque étape du parcours (ville i vers ville i+1)
    for i in range(len(parcours) - 1):
        ville_depart = parcours[i]
        ville_arrivee = parcours[i + 1]
        
        distance_totale += obtenir_distance(ville_depart, ville_arrivee, donnees)
        cout_total += obtenir_cout(ville_depart, ville_arrivee, donnees)
    
    # IMPORTANT: Ajouter le retour à la ville de départ
    derniere_ville = parcours[-1]
    premiere_ville = parcours[0]
    distance_totale += obtenir_distance(derniere_ville, premiere_ville, donnees)
    cout_total += obtenir_cout(derniere_ville, premiere_ville, donnees)
    
    # Mettre à jour la solution
    solution['distance'] = distance_totale  # f1
    solution['cout'] = cout_total            # f2
    solution['score'] = distance_totale + cout_total  # F = f1 + f2

In [ ]:
# Exemple: Évaluer la solution générée précédemment
evaluer_solution(exemple_solution, DONNEES)

print("Solution évaluée:")
print(f"Parcours: {exemple_solution['parcours']}")
print(f"Distance (f1): {exemple_solution['distance']}")
print(f"Coût (f2): {exemple_solution['cout']}")
print(f"Score (F = f1 + f2): {exemple_solution['score']}")

---

## 4. Opérateur de Croisement (Crossover)

### Croisement à Deux Points

Pour le croisement, deux solutions (parents) sont choisies, puis un **croisement à deux points** est appliqué.

### Procédure

1. Choisir deux points de coupure aléatoires (point1 et point2)
2. Copier le segment entre point1 et point2 du parent2 vers l'enfant
3. Identifier les villes qui apparaissent en double
4. Remplacer les doublons par les villes manquantes

### Exemple

```
P1 = [1 4 7 10 13 15 14 12 11 9 8 6 5 3 2]
P2 = [1 2 3 5  6  8  9 11 12 14 15 13 10 7 4]

Point1 = 5, Point2 = 10

Enfant avant correction: [1 4 7 10 13 | 8 9 11 12 14 | 8 6 5 3 2]
                                       ↑ segment de P2 ↑

Doublons: 8 apparaît deux fois
Manquantes: 15

Enfant après correction: [1 4 7 10 13 8 9 11 12 14 15 6 5 3 2]
```

In [ ]:
def croisement_deux_points(parent1, parent2, donnees):
    """
    Effectue un croisement à deux points entre deux parents.
    
    Paramètres:
        parent1: Première solution parent
        parent2: Deuxième solution parent
        donnees: Dictionnaire contenant les données du problème
    
    Retourne:
        Une nouvelle solution (enfant) issue du croisement
    """
    parcours1 = parent1['parcours']
    parcours2 = parent2['parcours']
    taille = len(parcours1)
    
    # Étape 1: Choisir deux points de coupure aléatoires
    point1 = random.randint(1, taille - 2)
    point2 = random.randint(point1 + 1, taille - 1)
    
    # Étape 2: Commencer avec le parcours du parent1
    nouveau_parcours = parcours1.copy()
    
    # Étape 3: Copier le segment du parent2 entre les deux points
    for i in range(point1, point2):
        nouveau_parcours[i] = parcours2[i]
    
    # Étape 4: Identifier les doublons
    villes_vues = set()
    positions_doublons = []
    
    for position, ville in enumerate(nouveau_parcours):
        if ville in villes_vues:
            positions_doublons.append(position)  # Cette position a un doublon
        else:
            villes_vues.add(ville)
    
    # Étape 5: Trouver les villes manquantes
    toutes_villes = set(donnees['villes'])
    villes_manquantes = list(toutes_villes - villes_vues)
    random.shuffle(villes_manquantes)  # Mélanger pour plus de diversité
    
    # Étape 6: Remplacer les doublons par les villes manquantes
    for i, position in enumerate(positions_doublons):
        nouveau_parcours[position] = villes_manquantes[i]
    
    # Créer et retourner la nouvelle solution
    enfant = creer_solution_vide()
    enfant['parcours'] = nouveau_parcours
    return enfant

---

## 5. Opérateur de Mutation

### Mutation par Échange

Une solution est sélectionnée aléatoirement pour appliquer une opération de **mutation par échange**.

### Procédure

1. Avec une probabilité égale au taux de mutation (15%)
2. Choisir deux positions i et j aléatoires
3. Échanger les villes à ces positions

### Exemple

```
Avant:  P = [1 4 7 10 13 15 14 12 11 9 8 6 5 3 2]
             ↑         ↑
Position i = 4 (ville 13)
Position j = 13 (ville 3)

Après:  R = [1 4 7 10 3 15 14 12 11 9 8 6 5 13 2]
```

In [ ]:
def mutation_echange(solution, taux_mutation):
    """
    Applique une mutation par échange de deux villes.
    
    La mutation n'est appliquée qu'avec une certaine probabilité
    (définie par le taux de mutation).
    
    Cette fonction modifie la solution en place.
    
    Paramètres:
        solution: La solution à muter
        taux_mutation: Probabilité d'effectuer la mutation (0.0 à 1.0)
    """
    # Vérifier si on effectue la mutation (selon le taux)
    if random.random() < taux_mutation:
        parcours = solution['parcours']
        taille = len(parcours)
        
        if taille >= 2:
            # Choisir deux positions différentes aléatoirement
            i = random.randint(0, taille - 1)
            j = random.randint(0, taille - 1)
            
            # S'assurer que i et j sont différents
            while j == i:
                j = random.randint(0, taille - 1)
            
            # Échanger les deux villes
            parcours[i], parcours[j] = parcours[j], parcours[i]

---

## 6. Sélection par Tournoi

### Principe

La sélection par tournoi choisit un individu pour la reproduction :

1. Sélectionner aléatoirement k individus de la population (k = taille du tournoi)
2. Retourner le meilleur parmi ces k individus

### Avantages

- Favorise les meilleures solutions sans exclure totalement les moins bonnes
- Permet de maintenir la diversité génétique

In [ ]:
def selection_tournoi(population, taille_tournoi):
    """
    Sélectionne une solution par tournoi.
    
    Paramètres:
        population: Liste de toutes les solutions
        taille_tournoi: Nombre de participants au tournoi
    
    Retourne:
        La meilleure solution parmi les participants du tournoi
    """
    # Choisir des participants aléatoires
    participants = random.sample(population, taille_tournoi)
    
    # Trouver le meilleur (score le plus bas = meilleur)
    meilleur = participants[0]
    for solution in participants[1:]:
        if solution['score'] < meilleur['score']:
            meilleur = solution
    
    return meilleur

---

## 7. Algorithme Génétique Principal

### Déroulement de l'algorithme

1. **Initialisation**: Générer une population de 300 solutions aléatoires

2. **Boucle principale** (500 itérations):
   - Évaluer toutes les solutions (calculer f1, f2, F)
   - Trier par score croissant (meilleur en premier)
   - Conserver la meilleure solution globale
   - Créer la nouvelle génération:
     - **Élitisme**: Garder le meilleur
     - Pour chaque nouvelle solution:
       - Soit croisement (70% des cas)
       - Soit copie simple (30% des cas)
       - Puis mutation (15% de chance)

3. **Résultat**: Retourner la meilleure solution trouvée

In [ ]:
def creer_population_initiale(taille, donnees):
    """
    Crée une population initiale de solutions aléatoires.
    
    Paramètres:
        taille: Nombre de solutions à créer
        donnees: Dictionnaire contenant les données du problème
    
    Retourne:
        Liste de solutions aléatoires
    """
    population = []
    for _ in range(taille):
        solution = creer_solution_aleatoire(donnees)
        population.append(solution)
    return population


def evaluer_population(population, donnees):
    """
    Évalue toutes les solutions d'une population.
    
    Paramètres:
        population: Liste des solutions
        donnees: Dictionnaire contenant les données du problème
    """
    for solution in population:
        evaluer_solution(solution, donnees)


def trier_population(population):
    """
    Trie la population par score croissant.
    Le meilleur (score le plus bas) sera en première position.
    
    Paramètres:
        population: Liste des solutions à trier
    """
    population.sort(key=lambda s: s['score'])


def creer_nouvelle_generation(population, donnees, params):
    """
    Crée la génération suivante à partir de la population actuelle.
    
    Paramètres:
        population: Population actuelle (triée)
        donnees: Dictionnaire contenant les données du problème
        params: Paramètres de l'algorithme
    
    Retourne:
        Nouvelle population
    """
    nouvelle_population = []
    
    # Élitisme: conserver le meilleur individu
    meilleur = copier_solution(population[0])
    nouvelle_population.append(meilleur)
    
    # Remplir le reste de la population
    while len(nouvelle_population) < params['taille_population']:
        
        # Décider si on fait un croisement ou une copie
        if random.random() < params['taux_croisement']:
            # Croisement entre deux parents
            parent1 = selection_tournoi(population, params['taille_tournoi'])
            parent2 = selection_tournoi(population, params['taille_tournoi'])
            enfant = croisement_deux_points(parent1, parent2, donnees)
        else:
            # Copie d'un parent sélectionné
            parent = selection_tournoi(population, params['taille_tournoi'])
            enfant = copier_solution(parent)
        
        # Appliquer la mutation
        mutation_echange(enfant, params['taux_mutation'])
        
        nouvelle_population.append(enfant)
    
    return nouvelle_population

In [ ]:
def executer_algorithme_genetique(donnees, params):
    """
    Exécute l'algorithme génétique complet.
    
    Paramètres:
        donnees: Dictionnaire contenant les données du problème
        params: Paramètres de l'algorithme
    
    Retourne:
        La meilleure solution trouvée
    """
    print(f"Démarrage de l'algorithme génétique")
    print(f"Population: {params['taille_population']} individus")
    print(f"Générations: {params['nb_generations']}")
    print("=" * 70)
    
    # Étape 1: Créer la population initiale
    population = creer_population_initiale(params['taille_population'], donnees)
    
    # Variables pour suivre le meilleur global
    meilleur_global = None
    generation_meilleur = 0
    
    # Étape 2: Boucle principale (500 itérations)
    for generation in range(params['nb_generations']):
        
        # Évaluer toutes les solutions
        evaluer_population(population, donnees)
        
        # Trier par score (meilleur en premier)
        trier_population(population)
        
        # Vérifier si on a trouvé un meilleur global
        champion = population[0]
        
        if meilleur_global is None or champion['score'] < meilleur_global['score']:
            meilleur_global = copier_solution(champion)
            generation_meilleur = generation
            print(f"Génération {generation:4d}: F={champion['score']} (dist={champion['distance']}, coût={champion['cout']})")
        
        # Créer la génération suivante
        population = creer_nouvelle_generation(population, donnees, params)
    
    print("=" * 70)
    print(f"\nMeilleure solution trouvée à la génération {generation_meilleur}")
    
    return meilleur_global

---

## 8. Fonctions d'Affichage et Export XML

Chaque solution doit être représentée par un fichier XML selon le format :

```xml
<?xml version="1.0" encoding="UTF-8"?>
<TSP_Solutions>
    <BestFitness>512</BestFitness>
    <TotalSolutions>2</TotalSolutions>
    
    <Solution id="1">
        <Tour>C6,C5,C4,C3,C1,C2,C10,C11,C14,C15,C13,C12,C9,C8,C7,C6</Tour>
        <Distance>204</Distance>
        <Cost>308</Cost>
    </Solution>
</TSP_Solutions>
```

In [ ]:
def afficher_solution(solution):
    """
    Affiche une solution de manière lisible.
    
    Paramètres:
        solution: La solution à afficher
    """
    parcours = solution['parcours']
    # Ajouter le retour à la ville de départ pour l'affichage
    trajet = " → ".join(parcours + [parcours[0]])
    
    print(f"Score F = {solution['score']} (Distance={solution['distance']}, Coût={solution['cout']})")
    print(f"Parcours: {trajet}")


def solution_vers_chaine(solution):
    """
    Convertit le parcours en chaîne CSV pour l'export XML.
    
    Paramètres:
        solution: La solution à convertir
    
    Retourne:
        Chaîne au format "C1,C4,C7,...,C1"
    """
    parcours = solution['parcours']
    return ",".join(parcours + [parcours[0]])

In [ ]:
def charger_solutions_xml(fichier):
    """
    Charge les solutions existantes depuis un fichier XML.
    
    Paramètres:
        fichier: Chemin vers le fichier XML
    
    Retourne:
        (meilleur_score, liste_parcours)
        ou (-1, []) si le fichier n'existe pas
    """
    if not os.path.exists(fichier):
        return -1, []
    
    try:
        with open(fichier, 'r', encoding='utf-8') as f:
            contenu = f.read()
        
        # Chercher le score avec une expression régulière
        import re
        match_score = re.search(r'<BestFitness>(\d+)</BestFitness>', contenu)
        if not match_score:
            match_score = re.search(r'<Fitness>(\d+)</Fitness>', contenu)
        
        if not match_score:
            return -1, []
        
        score = int(match_score.group(1))
        
        # Chercher tous les parcours
        parcours_list = re.findall(r'<Tour>([^<]+)</Tour>', contenu)
        
        return score, parcours_list
    
    except Exception as e:
        print(f"Erreur de lecture: {e}")
        return -1, []


def sauvegarder_solutions_xml(fichier, score, distance, cout, liste_parcours):
    """
    Sauvegarde les solutions dans un fichier XML.
    
    Paramètres:
        fichier: Chemin vers le fichier XML
        score: Meilleur score F
        distance: Distance correspondante
        cout: Coût correspondant
        liste_parcours: Liste des parcours à sauvegarder
    """
    # Construire le contenu XML ligne par ligne
    lignes = [
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<TSP_Solutions>',
        f'    <BestFitness>{score}</BestFitness>',
        f'    <TotalSolutions>{len(liste_parcours)}</TotalSolutions>',
        ''
    ]
    
    for i, parcours in enumerate(liste_parcours, 1):
        lignes.append(f'    <Solution id="{i}">')
        lignes.append(f'        <Tour>{parcours}</Tour>')
        lignes.append(f'        <Distance>{distance}</Distance>')
        lignes.append(f'        <Cost>{cout}</Cost>')
        lignes.append('    </Solution>')
        lignes.append('')
    
    lignes.append('</TSP_Solutions>')
    
    # Écrire dans le fichier
    with open(fichier, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lignes))
    
    print(f"Solution sauvegardée dans {fichier}")

---

## 9. Exécution de l'Algorithme

À chaque itération, le programme :
- Évalue les solutions à l'aide des fonctions f1 et f2
- Conserve la meilleure solution selon le critère F = f1 + f2
- Sauvegarde cette solution dans le fichier XML final

In [ ]:
# Nom du fichier XML pour sauvegarder les solutions
FICHIER_XML = "solution.xml"

# Charger les solutions précédentes (si elles existent)
meilleur_precedent, parcours_existants = charger_solutions_xml(FICHIER_XML)

if meilleur_precedent > 0:
    print(f"Solutions existantes trouvées!")
    print(f"Meilleur score précédent: {meilleur_precedent}")
    print(f"Nombre de parcours: {len(parcours_existants)}")
    print()

# Exécuter l'algorithme génétique
resultat = executer_algorithme_genetique(DONNEES, PARAMS)

# Afficher le résultat
print()
afficher_solution(resultat)

---

## 10. Sauvegarde du Résultat

Le programme sauvegarde le résultat selon les règles suivantes :

| Situation | Action |
|-----------|--------|
| Nouveau score < Meilleur précédent | Remplacer toutes les solutions |
| Nouveau score = Meilleur précédent | Ajouter si c'est un nouveau parcours |
| Nouveau score > Meilleur précédent | Conserver les solutions existantes |

In [ ]:
# Convertir le résultat en chaîne pour comparaison
nouveau_parcours = solution_vers_chaine(resultat)

print("=" * 70)

if meilleur_precedent < 0:
    # Première exécution - sauvegarder le résultat
    sauvegarder_solutions_xml(FICHIER_XML, resultat['score'], 
                              resultat['distance'], resultat['cout'], 
                              [nouveau_parcours])
    print("PREMIÈRE SOLUTION ENREGISTRÉE")

elif resultat['score'] < meilleur_precedent:
    # Nouveau record - remplacer toutes les anciennes solutions
    sauvegarder_solutions_xml(FICHIER_XML, resultat['score'],
                              resultat['distance'], resultat['cout'],
                              [nouveau_parcours])
    print(f"NOUVEAU RECORD! (Amélioration: {meilleur_precedent} → {resultat['score']})")

elif resultat['score'] == meilleur_precedent:
    # Même score - vérifier si c'est un nouveau parcours
    if nouveau_parcours not in parcours_existants:
        parcours_existants.append(nouveau_parcours)
        sauvegarder_solutions_xml(FICHIER_XML, resultat['score'],
                                  resultat['distance'], resultat['cout'],
                                  parcours_existants)
        print(f"NOUVEAU PARCOURS DÉCOUVERT (même score: {resultat['score']})")
        print(f"Total de parcours optimaux: {len(parcours_existants)}")
    else:
        print(f"PARCOURS DÉJÀ CONNU (score: {resultat['score']})")

else:
    # Score moins bon - ne pas sauvegarder
    print(f"PAS D'AMÉLIORATION")
    print(f"Score actuel: {resultat['score']}")
    print(f"Meilleur connu: {meilleur_precedent}")

print("=" * 70)

---

## 11. Visualisation des Solutions Optimales

In [ ]:
# Recharger et afficher toutes les solutions
score_final, tous_parcours = charger_solutions_xml(FICHIER_XML)

print("SOLUTIONS OPTIMALES TROUVÉES")
print("=" * 70)
print(f"Meilleur score (F = f1 + f2): {score_final}")
print(f"Nombre de parcours différents: {len(tous_parcours)}")
print("=" * 70)
print()

for i, parcours in enumerate(tous_parcours, 1):
    print(f"Parcours {i}:")
    print(f"  {parcours}")
    print()

---

## 12. Recherche Intensive (Exécutions Multiples)

Pour trouver plus de solutions optimales, on peut exécuter l'algorithme plusieurs fois.

Décommentez la dernière ligne pour lancer une recherche intensive.

In [ ]:
def recherche_intensive(nb_executions):
    """
    Lance plusieurs exécutions de l'algorithme pour trouver plus de solutions.
    
    Paramètres:
        nb_executions: Nombre d'exécutions à effectuer
    """
    print(f"RECHERCHE INTENSIVE: {nb_executions} exécutions")
    print("#" * 70)
    
    for i in range(nb_executions):
        print(f"\n{'='*70}")
        print(f"EXÉCUTION {i+1}/{nb_executions}")
        print(f"{'='*70}")
        
        # Charger l'état actuel
        meilleur_actuel, parcours_actuels = charger_solutions_xml(FICHIER_XML)
        
        # Exécuter l'algorithme
        resultat = executer_algorithme_genetique(DONNEES, PARAMS)
        nouveau_parcours = solution_vers_chaine(resultat)
        
        # Traiter le résultat
        if meilleur_actuel < 0 or resultat['score'] < meilleur_actuel:
            sauvegarder_solutions_xml(FICHIER_XML, resultat['score'],
                                      resultat['distance'], resultat['cout'],
                                      [nouveau_parcours])
            print(f"→ NOUVEAU RECORD: {resultat['score']}")
        elif resultat['score'] == meilleur_actuel and nouveau_parcours not in parcours_actuels:
            parcours_actuels.append(nouveau_parcours)
            sauvegarder_solutions_xml(FICHIER_XML, resultat['score'],
                                      resultat['distance'], resultat['cout'],
                                      parcours_actuels)
            print(f"→ NOUVEAU PARCOURS (total: {len(parcours_actuels)})")
        else:
            print(f"→ Pas d'amélioration")
    
    # Résumé final
    score_final, tous_parcours = charger_solutions_xml(FICHIER_XML)
    print(f"\n{'#'*70}")
    print(f"RÉSUMÉ FINAL")
    print(f"Meilleur score: {score_final}")
    print(f"Parcours trouvés: {len(tous_parcours)}")
    print(f"{'#'*70}")


# Décommenter la ligne suivante pour lancer 10 exécutions:
# recherche_intensive(10)